# Data preparation

In [40]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd

## Tram data

In [ ]:
df_tramway = pd.read_csv(r'../Data/tramway_indiv_15min.csv', index_col=0)

We drop rows with C3, RX and TS.
We keep only the T1 rows and the rest will serve to validate the model later.

In [ ]:
df_tramway_1 = df_tramway[df_tramway['COD_LIG_CLI']=='T1']
df_tramway_1.head()

In [ ]:
df_tramway_1 = df_tramway_1.drop(columns= ['NOM_HASTUS',
 'COD_UTE',
 'FLG_PNT_ARR',
 'FLG_PNT_DPO','IDT_LIE',
 'FLG_ADR_EN_FACE'])

df_tramway_1

In [ ]:
df_tramway_1['Day of week'] = pd.to_datetime(df_tramway_1['VAL_DATE']).dt.weekday
df_tramway_1['Day of month'] = pd.to_datetime(df_tramway_1['VAL_DATE']).dt.day
df_tramway_1['Month'] = pd.to_datetime(df_tramway_1['VAL_DATE']).dt.month

df_tramway_1

In [ ]:
df_tramway_1.to_csv(r'../Cleaned Data/df_tramway_1.csv', index=True)

## Now for the weather data

**Links to understand data** :  
DOC POUR LES WEATHER CODES : https://www.historique-meteo.net/assets/docs/weathercodes.txt  
COLONNE OPINION : avis météo de 0 à 4 (0 = très défavorable • 1 = défavorable • 2 = mitigé • 3 = favorable • 4 = très favorable).

### Grouping data and removing unwanted columns

In [32]:
df_weather_2019 = pd.read_csv(r'../Data/historique-meteo-lyon-2019.csv')
df_weather_2020 = pd.read_csv(r'../Data/historique-meteo-lyon-2020.csv')
df_weather_2020.head()

,DATE,MAX_TEMPERATURE_C,MIN_TEMPERATURE_C,WINDSPEED_MAX_KMH,TEMPERATURE_MORNING_C_6H,TEMPERATURE_NOON_C_12H,TEMPERATURE_EVENING_C_18H,PRECIP_TOTAL_DAY_MM,HUMIDITY_MAX_PERCENT,VISIBILITY_AVG_KM,...,TEMPERATURE_NIGHT_C_3H,WEATHER_CODE_NIGHT_3H,TEMPERATURE_9H,WEATHER_CODE_9H,TEMPERATURE_15H,WEATHER_CODE_15H,TEMPERATURE_21H,WEATHER_CODE_21H,TEMPERATURE_MIDNIGHT_0H,WEATHER_CODE_MIDNIGHT_0H
0,01-01-20,9,3,3,3,7,8,0.0,82,10.0,...,3,116,4,116,9,116,6,116,3,113
1,01-02-20,8,2,10,2,7,6,0.0,85,10.0,...,2,116,4,116,8,116,5,116,3,116
2,01-03-20,9,2,5,4,8,7,0.0,87,10.0,...,4,119,5,119,9,119,2,119,5,116
3,01-04-20,8,3,20,6,8,5,0.2,92,10.0,...,5,122,7,176,7,116,3,116,5,116
4,01-05-20,6,2,14,2,6,4,0.0,86,10.0,...,2,116,3,116,6,116,3,116,2,116


In [33]:
df_weather = pd.concat([df_weather_2019, df_weather_2020])
df_weather.columns

Index(['DATE', 'MAX_TEMPERATURE_C', 'MIN_TEMPERATURE_C', 'WINDSPEED_MAX_KMH',
       'TEMPERATURE_MORNING_C_6H', 'TEMPERATURE_NOON_C_12H',
       'TEMPERATURE_EVENING_C_18H', 'PRECIP_TOTAL_DAY_MM',
       'HUMIDITY_MAX_PERCENT', 'VISIBILITY_AVG_KM', 'PRESSURE_MAX_MB',
       'CLOUDCOVER_AVG_PERCENT', 'HEATINDEX_MAX_C', 'DEWPOINT_MAX_C',
       'WINDTEMP_MAX_C', 'WEATHER_CODE_MORNING_6H', 'WEATHER_CODE_NOON_12H',
       'WEATHER_CODE_EVENING_18H', 'TOTAL_SNOW_MM', 'UV_INDEX', 'SUNHOUR',
       'OPINION', 'SUNSET', 'SUNRISE', 'TEMPERATURE_NIGHT_C_3H',
       'WEATHER_CODE_NIGHT_3H', 'TEMPERATURE_9H', 'WEATHER_CODE_9H',
       'TEMPERATURE_15H', 'WEATHER_CODE_15H', 'TEMPERATURE_21H',
       'WEATHER_CODE_21H', 'TEMPERATURE_MIDNIGHT_0H',
       'WEATHER_CODE_MIDNIGHT_0H'],
      dtype='str')

In [34]:
df_weather.drop(columns=['MAX_TEMPERATURE_C', 'MIN_TEMPERATURE_C', 'WINDSPEED_MAX_KMH','PRECIP_TOTAL_DAY_MM','HUMIDITY_MAX_PERCENT', 'VISIBILITY_AVG_KM', 'PRESSURE_MAX_MB',
       'CLOUDCOVER_AVG_PERCENT', 'HEATINDEX_MAX_C', 'DEWPOINT_MAX_C',
       'WINDTEMP_MAX_C','TOTAL_SNOW_MM','UV_INDEX','SUNHOUR',
       'OPINION', 'SUNSET', 'SUNRISE'], inplace=True)
df_weather.head()

/var/folders/4h/ftzzlfmn3c9bnhtr2nlfsrb40000gn/T/ipykernel_54370/3706541768.py:1: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df_weather.drop(columns=['MAX_TEMPERATURE_C', 'MIN_TEMPERATURE_C', 'WINDSPEED_MAX_KMH','PRECIP_TOTAL_DAY_MM','HUMIDITY_MAX_PERCENT', 'VISIBILITY_AVG_KM', 'PRESSURE_MAX_MB',


,DATE,TEMPERATURE_MORNING_C_6H,TEMPERATURE_NOON_C_12H,TEMPERATURE_EVENING_C_18H,WEATHER_CODE_MORNING_6H,WEATHER_CODE_NOON_12H,WEATHER_CODE_EVENING_18H,TEMPERATURE_NIGHT_C_3H,WEATHER_CODE_NIGHT_3H,TEMPERATURE_9H,WEATHER_CODE_9H,TEMPERATURE_15H,WEATHER_CODE_15H,TEMPERATURE_21H,WEATHER_CODE_21H,TEMPERATURE_MIDNIGHT_0H,WEATHER_CODE_MIDNIGHT_0H
0,01-01-19,4,7,2,143,266,116,2,143,5,119,6,116,2,116,2,266
1,01-02-19,4,5,0,119,116,113,2,119,4,119,4,113,-1,113,1,116
2,01-03-19,-1,4,-2,113,113,113,-2,113,1,113,3,113,-3,116,-2,113
3,01-04-19,-1,5,-2,116,113,113,-3,113,1,113,3,113,-3,113,-3,113
4,01-05-19,0,3,0,332,122,122,-3,326,1,326,3,119,1,122,-4,119


### Making it fit the other dataset

In [42]:
# That part was made using Claude. I provided the structure of my file and asked the code to make it like an example I provided. 
# Map each 6-hour period start to its temperature and weather code columns
period_map = {
    0:  ('TEMPERATURE_MIDNIGHT_0H',  'WEATHER_CODE_MIDNIGHT_0H'),
    6:  ('TEMPERATURE_MORNING_C_6H', 'WEATHER_CODE_MORNING_6H'),
    12: ('TEMPERATURE_NOON_C_12H',   'WEATHER_CODE_NOON_12H'),
    18: ('TEMPERATURE_EVENING_C_18H', 'WEATHER_CODE_EVENING_18H'),
}

daily = df_weather.copy()
daily['DATE'] = pd.to_datetime(daily['DATE'], format='%m-%d-%y')

# Long table: one row per (day, period) with Temperature and weather_code
parts = []
for hour, (t_col, w_col) in period_map.items():
    part = daily[['DATE', t_col, w_col]].copy()
    part.columns = ['DAY', 'Temperature', 'weather_code']
    part['PERIOD'] = hour
    parts.append(part)
values = pd.concat(parts, ignore_index=True)

# Every 15 minutes for every day in the data
times = pd.date_range(
    start=daily['DATE'].min(),
    end=daily['DATE'].max() + pd.Timedelta(hours=23, minutes=45),
    freq='15min',
)
out = pd.DataFrame({'VAL_DATE': times})
out['DAY'] = out['VAL_DATE'].dt.normalize()
out['PERIOD'] = out['VAL_DATE'].dt.hour // 6 * 6

out = out.merge(values, on=['DAY', 'PERIOD'], how='left')
out = out[['VAL_DATE', 'Temperature', 'weather_code']]

# Format dates like the example (11/2/19 15:00)
# On Linux/macOS use %-m and %-d; on Windows use %#m and %#d
out['VAL_DATE'] = out['VAL_DATE'].dt.strftime('%-m/%-d/%y %H:%M')
out.to_csv('../Data/weather_15min_2019_2020.csv', index=False)
print(out.shape)
print(out.head())

(70176, 3)
       VAL_DATE  Temperature  weather_code
0  1/1/19 00:00            2           266
1  1/1/19 00:15            2           266
2  1/1/19 00:30            2           266
3  1/1/19 00:45            2           266
4  1/1/19 01:00            2           266


In [36]:
df_weather_daily = out.copy()

### Replacing weather codes
So now, we are going to convert weather codes in ice(3)/snow(2)/rain(1)/no rain no snow(0). using the table weather-codes.csv

In [37]:
weather_codes = pd.read_csv(r'../Cleaned Data/weather-codes.csv', index_col='WeatherCode')
weather_codes.columns


Index(['Condition', 'Associated number'], dtype='str')

In [41]:
df_weather_daily = df_weather_daily.merge(weather_codes, left_on='weather_code', right_index=True, how='left')
df_weather_daily.head()
df_weather_daily.to_csv(r'../Cleaned Data/weather_cleaned.csv',index=False)